# 02 - Feature Engineering

**Objetivo**: construir el dataset entrenable para el clasificador W/D/L (capa gratuita).

**Input**: Martj42 — 49.287 partidos internacionales (1872-2026).
**Output**: ~31.000 filas con 34 features numéricas, guardadas en `ml/data/processed/match_features.csv`.

**Decisión de diseño**: entrenamos con TODOS los partidos internacionales desde 1990, añadiendo el tipo de torneo como feature. Mantenemos el volumen completo en vez de filtrar solo a Mundiales (~1k partidos) porque el modelo aprende mejor con más muestras.

**Features construidas** (sin data leakage — todas usan información previa al partido):

1. **Rolling stats por equipo** (ventanas de 5 y 10 partidos previos): win_rate, avg goles a favor, avg goles en contra, avg diferencial.
2. **Rating Elo iterativo** con ajuste por margen de victoria y ventaja de localía.
3. **H2H** (head-to-head): % victorias en últimos 10 enfrentamientos directos entre los dos equipos.
4. **Descanso**: días desde el último partido de cada equipo.
5. **Sede neutral**: flag binario (ya en Martj42).
6. **Tipo de torneo agrupado**: mundial, eliminatoria, copa continental, amistoso, etc.


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]  # ml/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_international_results
from src.features import build_match_features, get_feature_columns, FeatureBuildConfig
from src.features import compute_elo_ratings, to_long_format

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
pd.set_option('display.max_columns', 50)


## 1. Cargar datos crudos

In [ ]:
matches = load_international_results()
print(f'Total partidos: {len(matches):,}')
print(f'Rango fechas: {matches.date.min().date()} → {matches.date.max().date()}')
print(f'\nDistribución del target:')
print(matches.result.value_counts())


## 2. Construir features

Una sola llamada hace todo el pipeline. Toma ~12 segundos en el dataset completo.


In [ ]:
config = FeatureBuildConfig(
    rolling_windows=(5, 10),
    h2h_window=10,
    min_year=1990,           # solo era moderna del fútbol internacional
    drop_first_n_per_team=5  # descartar primeros 5 partidos de cada equipo (sin historial)
)

%time features = build_match_features(matches, config)
print(f'\nShape: {features.shape}')


In [ ]:
# Inspeccionar columnas generadas
feature_cols = get_feature_columns()
print(f'Feature cols ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c}')


## 3. Validar calidad de las features

In [ ]:
# Distribución del target tras filtros
print('Distribución del target en el dataset entrenable:')
print(features.result.value_counts())
print('\nProporciones:')
print((features.result.value_counts(normalize=True) * 100).round(2).astype(str) + ' %')


In [ ]:
# Nulos en las features (los esperables son H2H sin historial y Elo del primer partido absoluto)
nas = features[feature_cols].isna().sum()
print('Features con valores nulos:')
print(nas[nas > 0])
print(f'\nTotal filas: {len(features):,}')


In [ ]:
# Stats descriptivas de features clave
key_features = ['team1_elo_before', 'team2_elo_before', 'diff_elo',
                'team1_roll10_win_rate', 'team2_roll10_win_rate',
                'team1_roll10_avg_gf', 'team2_roll10_avg_ga',
                'h2h_team1_win_rate', 'h2h_n_matches']
features[key_features].describe().round(3)


### 3.1 Visualización: Elo vs probabilidad de victoria observada

Validamos que el Elo es informativo: a mayor diferencia de Elo, mayor tasa de victoria del local.


In [ ]:
# Discretizar diff_elo en bins y ver tasa de victoria del local en cada bin
f = features.dropna(subset=['diff_elo']).copy()
f['elo_bin'] = pd.cut(f['diff_elo'], bins=np.arange(-500, 600, 100))

win_rate_by_elo = f.groupby('elo_bin', observed=True).agg(
    home_win_rate=('result', lambda s: (s == 'home_win').mean()),
    draw_rate=('result', lambda s: (s == 'draw').mean()),
    away_win_rate=('result', lambda s: (s == 'away_win').mean()),
    n=('result', 'count'),
).round(3)
print(win_rate_by_elo)

fig, ax = plt.subplots()
win_rate_by_elo[['home_win_rate', 'draw_rate', 'away_win_rate']].plot(kind='bar', stacked=True, ax=ax,
    color=['#2E86AB', '#F18F01', '#A23B72'])
ax.set_title('Tasa de resultado real según diferencia de Elo (team1 - team2)')
ax.set_xlabel('Bin de diff_elo'); ax.set_ylabel('Proporción')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout(); plt.show()


### 3.2 Correlación entre features y target

In [ ]:
# Encodear target a numérico para mostrar correlaciones lineales
f = features.dropna(subset=feature_cols).copy()
f['target_num'] = f['result'].map({'home_win': 1, 'draw': 0, 'away_win': -1})

corrs = f[feature_cols + ['target_num']].corr()['target_num'].drop('target_num').sort_values(key=abs, ascending=False)
print('Correlación de cada feature con el resultado (1=home_win, -1=away_win):')
print(corrs.round(3))

fig, ax = plt.subplots(figsize=(8, 10))
corrs.plot(kind='barh', ax=ax, color=['#2E86AB' if v>0 else '#A23B72' for v in corrs])
ax.set_title('Correlación de cada feature con el target')
ax.invert_yaxis()
plt.tight_layout(); plt.show()


## 4. Tipo de torneo agrupado

In [ ]:
tg_counts = features['tournament_group'].value_counts()
print('Distribución por tipo de torneo agrupado:')
print(tg_counts)
print()

# Tasa de empate por tipo de torneo (los empates son más raros en partidos importantes)
draw_rate_by_tg = features.groupby('tournament_group').apply(
    lambda d: (d.result == 'draw').mean()).round(3).sort_values(ascending=False)
print('Tasa de empate por tipo de torneo:')
print(draw_rate_by_tg)


## 5. Guardar dataset procesado

Persistimos en CSV para uso por los notebooks de modelado.


In [ ]:
out_path = ROOT / 'data' / 'processed' / 'match_features.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)

# Columnas a persistir: features + target + metadata identificativa
keep_cols = (
    ['date', 'home_team', 'away_team', 'home_score', 'away_score',
     'tournament', 'tournament_group', 'neutral', 'result']
    + feature_cols
)

features[keep_cols].to_csv(out_path, index=False)
print(f'✓ Guardado en {out_path.relative_to(ROOT.parent)}')
print(f'  Filas: {len(features):,}')
print(f'  Tamaño: {out_path.stat().st_size / 1024:.1f} KB')


## Próximos pasos

1. **Notebook `03_classification_model.ipynb`**: cargar `match_features.parquet`, entrenar baseline + Random Forest + XGBoost + Regresión Logística, comparar con validación cruzada y persistir el mejor modelo en `ml/trained_models/`.
2. **Notebook `04_regression_models.ipynb`**: modelos premium (goles 1T/2T desde Fjelstul, corners/posesión/faltas desde WC2022).
3. **Notebook `05_clustering.ipynb`**: K-Means para clasificar selecciones por estilo de juego.
